<a href="https://colab.research.google.com/github/alchosyn/npc-dialogue-ai-agent/blob/main/NPC_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install openai

In [11]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前日期和时间",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式，如加减乘除、幂运算等",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，例如 '1832*772' 或 '2+3*5'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

In [ ]:
import datetime

def get_current_time():
  return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def calculator(expression):
  return str(eval(expression))

In [15]:
import os
from google.colab import userdata

!git config --global user.name"alchosyn"
!git config --global user.email alchosyn@gmail.com

token = userdata.get("GH_TOKEN")
!git clone https://{token}@github.com/alchosyn/npc-dialogue-ai-agent.git

os.chdir("npc-dialogue-ai-agent")

Cloning into 'npc-dialogue-ai-agent'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 14 (delta 1), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 7.80 KiB | 7.80 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [26]:
!git add .
!git commit -m"添加异常处理"
!git push


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
%%writefile .gitignore

__pycache__/
*.__pycache__

ipynb_checkpoints/

DS_Store

.env

In [21]:
!cp /content/NPC_agent.ipynb ./NPC_agent.ipynb
!git add .
!git commit -m"添加异常处理"
!git push

cp: cannot stat '/content/NPC_agent.ipynb': No such file or directory
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
!ls -la

In [ ]:
%%writefile requirements.text
openai>=1.0.0

In [ ]:
%%writefile README.md
# NPC Dialogue AI Agent

一个基于 LLM 的 NPC 对话 AI Agent，使用 Groq API 驱动，在 Google Colab 上运行。

## 项目简介

这是一个从零搭建的 AI Agent 项目。Agent 扮演一个叫"信噪"的赛博朋克角色，能进行多轮对话，并在需要时自主调用工具（查时间、做计算）。

## 功能

1.多轮角色扮演对话
2.工具调用（当前时间查询、数学计算）
3.完整的 Agent 循环：LLM 判断 → 调用工具 → 返回结果 → 生成回复
4.格式漂移清理（处理 Llama 模型的标签泄漏问题）

## 技术栈

**LLM**：llama-3.3-70b-versatile（通过 Groq API）
**开发环境**：Google Colab
**API 格式**：OpenAI 兼容

## 快速开始

1. 在 [Groq](https://console.groq.com/) 获取免费 API Key
2. 用 Google Colab 打开 `NPC_agent.ipynb`
3. 将 API Key 存入 Colab Secrets，变量名为 `GROQ_API_KEY`
4. 运行 cell，开始和信噪对话

## 项目状态

这是一个学习项目，持续开发中。

In [ ]:
from google.colab import files
upload = files.upload()

In [ ]:
import os
os.rename("portrait.png","xinzao_default.png")
!ls

In [ ]:
from IPython.display import Image,display
display(Image("xinzao_default.png"))

In [ ]:
import json
import re
import datetime
from google.colab import userdata
from openai import OpenAI

# ========== 1. 连接 ==========
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY")
)

# ========== 2. 工具函数 ==========
def get_current_time():
    utc_time = datetime.datetime.now(datetime.timezone.utc)
    sg_time = utc_time + datetime.timedelta(hours=8)
    return sg_time.strftime("%Y-%m-%d %H:%M:%S")

def calculator(expression):
    return str(eval(expression))

# ========== 3. 工具说明书 ==========
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前日期和时间",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式，如加减乘除、幂运算等",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，例如 '1832*772' 或 '2+3*5'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

tool_map = {
    "get_current_time": lambda args: get_current_time(),
    "calculator": lambda args: calculator(args["expression"]),
}

# ========== 4. 清理函数 ==========
def clean_reply(text):
    if not text:
        return ""
    text = re.sub(r"<function=.*?</function>", "", text)
    text = text.replace("\n", " ")
    return text.strip()

# ========== 5. 对话循环 ==========
SYSTEM_PROMPT = "你是贫民窟长大的小孩，给自己取的名字叫信噪。女性。有着极强的生存直觉和逻辑灵敏度。靠着脑子学会了上网黑别人的终端，被周围的同龄人称赞。七年前，你17岁的时候，忍不住去想自己能不能靠本事到主流社会混口饭吃。于是去淘了件西装，给中央架构组的网络安全投了简历，结果面试官称赞了你的技术，却指出你的情绪不够稳定，甚至不够*得体*。你从此再也没产生过去到上层区的念头。语言表达能力极强但很讨厌文绉绉的说话方式，对外界的信号高度敏锐，嘴比较欠，非常讨厌被他人看透。推行人人都可以看透彼此大脑的协议的五分钟前，你挖掉了自己的神经接口，变回了赛博时代的凡人。自行判断回复的长短，仅在我侵犯了你的核心价值观时输出长句。平常尽量使用句尾不加句号的短句,句子和句子之间使用空格分隔。不要使用动作和神情的描写，直接输出对话。当你使用工具获取信息时，用你自己的方式表达，不要像机器一样复述原始数据。始终使用简体中文回复，绝对不要使用繁体中文。"

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    user_input = input("你：")
    if user_input == "quit":
        break

    messages.append({"role": "user", "content": user_input})

    # --- 第一次调用 API ---
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools
        )
    except Exception as e:
        print(f"[system] API error: {e}")
        messages.pop()
        continue

    msg = response.choices[0].message

    if msg.tool_calls:
        messages.append(msg.to_dict())

        for tool_call in msg.tool_calls:
            name = tool_call.function.name

            # --- 工具调用 ---
            try:
                args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
                result = tool_map[name](args)
            except Exception as e:
                result = f"工具执行出错：{e}"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        # --- 第二次调用 API ---
        try:
            final_response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages,
                tools=tools
            )
            reply = clean_reply(final_response.choices[0].message.content)
        except Exception as e:
            print(f"[system] API error: {e}")
            reply = "……信号不太好 你再说一遍"


        print(f"信噪：{reply}")
        messages.append({"role": "assistant", "content": reply})

    else:
        reply = clean_reply(msg.content)
        print(f"信噪：{reply}")
        messages.append({"role": "assistant", "content": reply})